In [ ]:
#@title 1. Calibration / Калибровка { display-mode: "form" }
#@markdown Fill in the form, then press play. Nothing before this cell needs running.
#@markdown ---
USE_EXAMPLE_DATA = True #@param {type:"boolean"}
#@markdown Tick the box to try the whole pipeline on the bundled example instead of your own data.
REPETITIONS = 5000 #@param {type:"integer"}
SEED = 20260719 #@param {type:"integer"}
SCENARIO = "location" #@param ["location", "decrease", "variability"]
ALPHA = 0.05 #@param {type:"number"}
EFFECT = 1.15 #@param {type:"number"}
SPLIT_CALIBRATION = False #@param {type:"boolean"}
PROJECT = "Colab run" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

import os, subprocess, sys, time, glob, zipfile, shutil, textwrap, shlex

REPO_URL = "https://github.com/d1d2dopamine/MVS-Analyzer.git"
ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
SRC = os.path.join(ROOT, "MVS-Analyzer")
BIN = os.path.join(ROOT, "mvs-bin")
OUT = os.path.join(ROOT, "mvs-out")
DOTNET_DIR = os.path.join(ROOT, "dotnet")
DOTNET = os.path.join(DOTNET_DIR, "dotnet")
MVS = os.path.join(BIN, "mvs")

def q(value):
    return shlex.quote(str(value))

def run(command, cwd=None, quiet=False):
    if not quiet:
        print("$ " + command)
    finished = subprocess.run(command, shell=True, cwd=cwd, text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if finished.stdout:
        print(finished.stdout.rstrip())
    if finished.returncode != 0:
        raise SystemExit("The step above failed with exit code %d." % finished.returncode)
    return finished.stdout

def step(number, text):
    print("")
    print("[%s] %s" % (number, text))

start = time.time()

step(1, "Fetching the source")
if not os.path.isdir(SRC):
    run("git clone --depth 1 --branch %s %s %s" % (q(BRANCH), q(REPO_URL), q(SRC)))
else:
    print("already present: " + SRC)

step(2, "Installing the .NET 8 build tools")
os.environ["DOTNET_CLI_TELEMETRY_OPTOUT"] = "1"
os.environ["DOTNET_NOLOGO"] = "1"
os.environ["DOTNET_SKIP_FIRST_TIME_EXPERIENCE"] = "1"
if not os.path.exists(DOTNET):
    print("this takes a minute or two, and only on the first run of a session")
    run("curl -sSL https://dot.net/v1/dotnet-install.sh -o /tmp/dotnet-install.sh")
    run("bash /tmp/dotnet-install.sh --channel 8.0 --install-dir %s --no-path" % q(DOTNET_DIR))
else:
    print("already present: " + DOTNET)

step(3, "Building the headless analyzer")
if not os.path.exists(MVS):
    run("%s publish %s -c Release -o %s --nologo -v minimal"
        % (q(DOTNET), q(os.path.join(SRC, "MvsAnalyzer.Cli", "MvsAnalyzer.Cli.csproj")), q(BIN)))
else:
    print("already built: " + MVS)
run("chmod +x " + q(MVS), quiet=True)
run("%s version" % q(MVS))
run("%s env" % q(MVS))

step(4, "Choosing the data")
JOB = ""
DATA = ""
if USE_EXAMPLE_DATA:
    DATA = os.path.join(SRC, "examples", "demo_three_groups.csv")
    print("using the bundled example: " + DATA)
else:
    print(textwrap.dedent("""
    Upload either a CSV of measurements, or the job archive the desktop app builds under
    "Remote run". The archive carries the settings with it, so the remote run is the same
    analysis as the one set up on your own machine rather than a similar one.

    Anything uploaded here leaves your computer and is processed on rented hardware. If these
    measurements are identifiable or restricted, close this notebook and run the analysis
    locally; the desktop app does the same work offline.
    """).strip())
    print("")
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit("This cell can only upload files inside Colab. On Kaggle, attach the data"
                         " as a dataset and set DATA to a path under /kaggle/input, or tick"
                         " USE_EXAMPLE_DATA.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("Nothing was uploaded.")
    name = list(uploaded.keys())[0]
    if name.lower().endswith(".zip"):
        folder = os.path.join(ROOT, "job")
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile(name) as archive:
            archive.extractall(folder)
        found = glob.glob(os.path.join(folder, "**", "job.json"), recursive=True)
        if found:
            JOB = found[0]
            print("job file: " + JOB)
        csvs = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
        if not csvs:
            raise SystemExit("The archive contained no CSV file.")
        DATA = csvs[0]
    else:
        DATA = os.path.join(ROOT, name) if os.path.exists(os.path.join(ROOT, name)) else name
    print("data: " + DATA)

step(5, "Calibrating")
CALIBRATION = os.path.join(OUT, "calibration")
os.makedirs(CALIBRATION, exist_ok=True)

if JOB:
    command = "%s calibrate --job %s --in %s --out %s" % (q(MVS), q(JOB), q(DATA), q(CALIBRATION))
else:
    command = ("%s calibrate --in %s --out %s --repetitions %d --seed %d --scenario %s"
               " --alpha %s --effect %s"
               % (q(MVS), q(DATA), q(CALIBRATION), int(REPETITIONS), int(SEED),
                  q(SCENARIO), q(ALPHA), q(EFFECT)))
    if SPLIT_CALIBRATION:
        command += " --split"
run(command)

print("")
print("Calibration finished in %d seconds. Now run the second cell." % int(time.time() - start))

In [ ]:
#@title 2. Analysis / Анализ { display-mode: "form" }
#@markdown Applies the calibration from the first cell. Run the first cell before this one.
MARGIN = 0.147 #@param {type:"number"}

ANALYSIS = os.path.join(OUT, "analysis")
os.makedirs(ANALYSIS, exist_ok=True)

run("%s analyze --in %s --calibration %s --out %s --project %s --margin %s"
    % (q(MVS), q(DATA), q(CALIBRATION), q(ANALYSIS), q(PROJECT), q(MARGIN)))

found = sorted(glob.glob(os.path.join(ANALYSIS, "**", "results.csv"), recursive=True))
if found:
    try:
        import pandas
        table = pandas.read_csv(found[-1])
        columns = [c for c in ["metric", "verdict", "global_p", "effect_cliffs_delta",
                               "calibrated_fpr", "candidate_tracks", "mvs_score"]
                   if c in table.columns]
        print("")
        print("results.csv")
        display(table[columns] if columns else table)
    except Exception as error:
        print("Could not render the table (%s). The file itself is written." % error)
    print("")
    print("Now run the third cell to download everything.")
else:
    print("No results.csv was found. Read the log above before continuing.")

In [ ]:
#@title 3. Pack the results / Упаковать результаты
import hashlib, datetime

stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S")
target = "/kaggle/working" if os.path.isdir("/kaggle/working") else ROOT
made = shutil.make_archive(os.path.join(target, "MVS_results_" + stamp), "zip", OUT)

print("archive   " + made)
print("size      %.1f KB" % (os.path.getsize(made) / 1024.0))
print("sha256    " + hashlib.sha256(open(made, "rb").read()).hexdigest())
print("")
print("On Kaggle the archive appears under Output on the right. Press Save Version first, then")
print("download it from the Output tab. A session needs Internet switched on in the settings")
print("panel, otherwise the first cell cannot fetch the source.")